# RT-DETR COCO 공통 평가 (Colab)
양자화는 로컬에서 수행합니다. 이 노트북은 **FP32 기준 모델과 로컬에서 만든 ONNX 모델 한 개**를 동일한 COCO val2017 1,000장에 대해 CPU로 평가합니다. COCO calibration 512장은 로컬 양자화용이며 이 노트북에서 양자화하지 않습니다.

**제출 모델 조건:** 단일 `.onnx` 파일, 입력 `images` (1×3×640×640 float32)와 `orig_target_sizes` (1×2 int64), 출력 `labels`·`boxes`·`scores` (각 300개 예측), COCO 80개 클래스. 출처 메타데이터가 없는 모델은 시험 형식 검증 후 팀 선언 모델로 등록됩니다. 등록은 학습 출처나 양자화 비트를 증명하지 않습니다.

In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/haiteam4_finetune')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/foxyaef/haiteam4_finetune.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-r', 'requirements-onnx.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Evaluation code commit:', commit)

## 공통 모델·데이터 준비
공식 COCO 사전학습 체크포인트와 고정 COCO 부분집합을 다운로드합니다. 데이터 압축 파일 약 1.1GB를 내려받고 1,512장만 추출합니다. 모델 평가에는 그중 1,000장만 사용합니다.

In [ ]:
subprocess.run([sys.executable, 'scripts/colab_setup.py'], check=True)
subprocess.run([sys.executable, 'prepare_data.py'], check=True)
subprocess.run([sys.executable, 'prepare_data.py', '--verify-only'], check=True)
fp32 = repo / 'artifacts/rtdetr_fp32.onnx'
if not fp32.exists():
    subprocess.run([sys.executable, 'scripts/onnx_real_ptq.py', 'export', '--output', str(fp32)], check=True)
print('FP32 baseline:', fp32, fp32.stat().st_size / 2**20, 'MiB')

## 로컬에서 양자화한 모델 업로드
아래 `method_name`을 실험 이름으로 바꾸고 ONNX 파일 **한 개**를 업로드하세요. 프로젝트의 로컬 PTQ 스크립트가 만든 모델은 기존 메타데이터를 사용합니다. 다른 도구로 만든 모델은 공통 입출력 형식을 검사해 등록하며, 출처와 방법은 팀 선언으로 표시합니다. 검증에 실패하면 그대로 중단합니다.

In [ ]:
method_name = 'my_local_ptq'  # 예: int8_static, brecq_backbone
from google.colab import files
sys.path.insert(0, str(repo / 'scripts'))
from onnx_graph import model_properties
upload_dir = repo / 'uploads'
upload_dir.mkdir(exist_ok=True)
os.chdir(upload_dir)
uploaded = files.upload()
os.chdir(repo)
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.onnx'):
    raise ValueError('단일 .onnx 파일을 업로드하세요')
quantized = upload_dir / next(iter(uploaded))
del uploaded
properties = model_properties(quantized)
if properties.get('ptq_kind') == 'onnx_fp32':
    raise ValueError('양자화된 ONNX 모델을 업로드하세요. FP32 기준 모델은 위에서 준비했습니다')
if properties.get('ptq_kind') not in ('onnx_fp32', 'onnx_int8', 'onnx_int4', 'onnx_experiment'):
    registered = repo / 'artifacts/registered_team_model.onnx'
    if registered.exists():
        registered.unlink()  # 재실행 시 이전 등록 파일만 교체
    subprocess.run([sys.executable, 'scripts/register_model.py', '--model', str(quantized),
                    '--baseline', str(fp32), '--output', str(registered), '--method', method_name], check=True)
    quantized = registered
print('Evaluation model:', quantized)
print('Identity:', model_properties(quantized).get('identity_status', 'project_generated'))

## 정확도·크기·속도·메모리 평가
FP32와 업로드 모델을 별도 프로세스로 평가합니다. ONNX Runtime **CPUExecutionProvider**만 사용하며 평가 이미지 1,000장, batch 1, warmup 10장입니다. 코랩 CPU의 종류는 세션마다 다를 수 있으므로 속도는 같은 세션 안에서 비교하세요.

In [ ]:
import datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
run_ids = {'fp32': 'onnx_fp32_' + stamp, 'experiment': 'onnx_experiment_' + stamp}
for name, model in [('fp32', fp32), ('experiment', quantized)]:
    subprocess.run([sys.executable, 'scripts/onnx_eval.py', '--model', str(model),
                    '--run-id', run_ids[name]], check=True)
print('Completed:', run_ids)

## 결과 다운로드
결과 ZIP을 로컬로 받아 `python scripts/analyze_real_ptq.py <ZIP 파일>`로 비교하세요. 정확도는 COCO 1,000장 부분집합 기준입니다. 제출 모델의 실제 양자화 범위는 ZIP의 `provenance.json` 연산 수와 원본 모델 그래프를 확인해야 합니다.

In [ ]:
import zipfile
bundle = repo / ('coco_eval_results_' + stamp + '.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for run_id in run_ids.values():
        for name in ('metrics.json', 'class_metrics.csv', 'profile.json', 'provenance.json'):
            result = repo / 'runs/onnx_eval' / run_id / name
            zipped.write(result, arcname=run_id + '/' + name)
files.download(str(bundle))
print('Saved:', bundle.name)